# TP 0 — Understanding Support Vector Machines (SVM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp_0_svm.ipynb)

## Objectives
1. Understand the intuition behind maximum margin classifiers
2. Learn the mathematical formulation of linear SVMs
3. Explore the effect of the regularization parameter $C$ (soft margin)
4. Understand the kernel trick for non-linear decision boundaries
5. Compare common kernels (linear, RBF, polynomial)
6. Explore the effect of the $\gamma$ parameter for the RBF kernel
7. Apply SVMs to regression with SVR

## Setup

Run the cell below to install and import the required packages.

In [ ]:
!pip install git+https://github.com/racousin/L2Math.git

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D
from sklearn.svm import SVC, SVR
from sklearn.datasets import make_blobs, make_classification, make_circles, make_moons
from sklearn.preprocessing import StandardScaler

from l2math import plot_decision_boundary_2d

np.random.seed(42)
print("Setup complete!")

---
# Part 1: Intuition — Maximum Margin Classifier

Given a binary classification problem with linearly separable data, there are **many possible separating hyperplanes** that perfectly classify the training points.

Which one should we pick?

The SVM answer: **the one that maximizes the margin** — the distance between the decision boundary and the closest data points from each class. This gives the classifier the best generalization to unseen data.

In [ ]:
# Generate linearly separable 2D data
X, y = make_blobs(n_samples=100, centers=2, cluster_std=0.8, random_state=42)

# Plot the data with several possible separating lines
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: multiple possible separating lines
ax = axes[0]
colors = ['#FF0000', '#0000FF']
for label in [0, 1]:
    mask = y == label
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[label], edgecolors='k',
               s=50, linewidths=0.5, label=f'Class {label}')

# Compute midpoint between cluster centers for drawing lines
center0 = X[y == 0].mean(axis=0)
center1 = X[y == 1].mean(axis=0)
midpoint = (center0 + center1) / 2

# Draw several possible separating lines
x_range = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
slopes = [-0.3, -0.6, -1.0, -1.5]
line_colors = ['green', 'orange', 'purple', 'brown']
for slope, lc in zip(slopes, line_colors):
    intercept = midpoint[1] - slope * midpoint[0]
    y_line = slope * x_range + intercept
    ax.plot(x_range, y_line, color=lc, linewidth=1.5, linestyle='--', alpha=0.7)

ax.set_xlim(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
ax.set_ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
ax.set_title('Many possible separating lines', fontsize=13)
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# Right plot: SVM max-margin boundary with margin band and support vectors
ax = axes[1]
svm = SVC(kernel='linear', C=1e6)
svm.fit(X, y)

# Extract parameters
w = svm.coef_[0]
b = svm.intercept_[0]
margin = 2 / np.linalg.norm(w)

# Decision boundary: w[0]*x + w[1]*y + b = 0  =>  y = -(w[0]*x + b) / w[1]
x_boundary = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
y_boundary = -(w[0] * x_boundary + b) / w[1]
y_margin_pos = -(w[0] * x_boundary + b - 1) / w[1]
y_margin_neg = -(w[0] * x_boundary + b + 1) / w[1]

# Scatter data
for label in [0, 1]:
    mask = y == label
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[label], edgecolors='k',
               s=50, linewidths=0.5, label=f'Class {label}')

# Decision boundary and margin lines
ax.plot(x_boundary, y_boundary, 'k-', linewidth=2, label='Decision boundary')
ax.plot(x_boundary, y_margin_pos, 'k--', linewidth=1, alpha=0.7, label='Margin boundary')
ax.plot(x_boundary, y_margin_neg, 'k--', linewidth=1, alpha=0.7)
ax.fill_between(x_boundary, y_margin_neg, y_margin_pos, alpha=0.15, color='yellow', label='Margin band')

# Highlight support vectors
ax.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
           s=200, facecolors='none', edgecolors='black', linewidths=2,
           label='Support vectors')

ax.set_xlim(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
ax.set_ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
ax.set_title(f'SVM: Maximum margin (width = {margin:.2f})', fontsize=13)
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Number of support vectors: {len(svm.support_vectors_)}")
print(f"Margin width: {margin:.4f}")

---
# Part 2: The Math — Linear SVM

A linear SVM finds the hyperplane that **maximizes the margin** between two classes.

### Decision Function

The decision function is:

$$f(x) = w^T x + b$$

A point $x$ is classified as:
- Class $+1$ if $f(x) \geq 0$
- Class $-1$ if $f(x) < 0$

### Optimization Problem

The SVM solves:

$$\min_{w, b} \frac{1}{2} \|w\|^2 \quad \text{subject to} \quad y_i(w^T x_i + b) \geq 1 \;\; \forall i$$

### Key Properties

- **Margin width** $= \frac{2}{\|w\|}$ : minimizing $\|w\|^2$ maximizes the margin
- **Support vectors**: the data points that lie exactly on the margin boundary, i.e., $y_i(w^T x_i + b) = 1$. These are the only points that influence the decision boundary.

In [ ]:
# Detailed visualization of linear SVM geometry
X_sep, y_sep = make_blobs(n_samples=80, centers=2, cluster_std=0.8, random_state=42)

# Train a hard-margin SVM (very large C)
svm_linear = SVC(kernel='linear', C=1e6)
svm_linear.fit(X_sep, y_sep)

# Extract model parameters
w = svm_linear.coef_[0]
b = svm_linear.intercept_[0]
w_norm = np.linalg.norm(w)
margin = 2.0 / w_norm

print(f"Weight vector w = [{w[0]:.4f}, {w[1]:.4f}]")
print(f"Bias b = {b:.4f}")
print(f"||w|| = {w_norm:.4f}")
print(f"Margin width = 2/||w|| = {margin:.4f}")
print(f"Number of support vectors: {len(svm_linear.support_vectors_)}")

# Create the plot
fig, ax = plt.subplots(figsize=(9, 7))

# Plot data points
cmap_bold = ListedColormap(['#FF0000', '#0000FF'])
ax.scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, cmap=cmap_bold,
           edgecolors='k', s=60, linewidths=0.5, zorder=3)

# Decision boundary line: w[0]*x1 + w[1]*x2 + b = 0
x_plot = np.linspace(X_sep[:, 0].min() - 1, X_sep[:, 0].max() + 1, 200)
y_decision = -(w[0] * x_plot + b) / w[1]
y_margin_upper = -(w[0] * x_plot + b - 1) / w[1]
y_margin_lower = -(w[0] * x_plot + b + 1) / w[1]

# Draw boundary and margin lines
ax.plot(x_plot, y_decision, 'k-', linewidth=2.5, label=r'$w^T x + b = 0$ (decision boundary)')
ax.plot(x_plot, y_margin_upper, 'k--', linewidth=1.5, alpha=0.7, label=r'$w^T x + b = +1$ (margin)')
ax.plot(x_plot, y_margin_lower, 'k--', linewidth=1.5, alpha=0.7, label=r'$w^T x + b = -1$ (margin)')

# Shade the margin band
ax.fill_between(x_plot, y_margin_lower, y_margin_upper,
                alpha=0.15, color='gold', label=f'Margin band (width = {margin:.2f})')

# Highlight support vectors
svs = svm_linear.support_vectors_
ax.scatter(svs[:, 0], svs[:, 1], s=250, facecolors='none',
           edgecolors='black', linewidths=2.5, zorder=4, label='Support vectors')

# Draw the w vector (normal to decision boundary)
mid_x = x_plot[len(x_plot)//2]
mid_y = y_decision[len(y_decision)//2]
scale = 0.5
ax.annotate('', xy=(mid_x + scale*w[0]/w_norm, mid_y + scale*w[1]/w_norm),
            xytext=(mid_x, mid_y),
            arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.text(mid_x + scale*w[0]/w_norm + 0.1, mid_y + scale*w[1]/w_norm + 0.1,
        r'$\mathbf{w}$', fontsize=14, color='green', fontweight='bold')

ax.set_xlim(X_sep[:, 0].min() - 1, X_sep[:, 0].max() + 1)
ax.set_ylim(X_sep[:, 1].min() - 1, X_sep[:, 1].max() + 1)
ax.set_title('Linear SVM: Decision Boundary, Margin, and Support Vectors', fontsize=14)
ax.set_xlabel('Feature 1', fontsize=12)
ax.set_ylabel('Feature 2', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 3: Soft Margin — The C Parameter

Real-world data is rarely perfectly linearly separable. The **soft margin SVM** allows some points to violate the margin or even be misclassified, by introducing slack variables $\xi_i \geq 0$.

The optimization becomes:

$$\min_{w, b, \xi} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^{n} \xi_i \quad \text{subject to} \quad y_i(w^T x_i + b) \geq 1 - \xi_i$$

The parameter $C$ controls the trade-off:
- **Large $C$**: penalizes misclassifications heavily $\rightarrow$ narrow margin, fewer margin violations
- **Small $C$**: allows more misclassifications $\rightarrow$ wide margin, more regularization

In [ ]:
# Generate noisy (not perfectly separable) 2D data
X_noisy, y_noisy = make_classification(
    n_samples=100, n_features=2, n_redundant=0, n_informative=2,
    random_state=42, n_clusters_per_class=1
)

# Compare three values of C
C_values = [0.01, 1, 100]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, C in zip(axes, C_values):
    svm_c = SVC(kernel='linear', C=C)
    svm_c.fit(X_noisy, y_noisy)
    n_sv = svm_c.n_support_.sum()
    plot_decision_boundary_2d(svm_c, X_noisy, y_noisy,
                              title=f'C = {C} ({n_sv} support vectors)', ax=ax)

plt.suptitle('Effect of C on the Linear SVM Decision Boundary', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- $C = 0.01$: wide margin, many support vectors, smooth boundary (high regularization)
- $C = 1$: balanced trade-off
- $C = 100$: narrow margin, few support vectors, boundary tightly fits the training data

---
# Part 4: Non-Linear Data — The Kernel Trick

Some datasets cannot be separated by **any** line (or hyperplane) in the original feature space. The idea: **map the data to a higher-dimensional space** where it becomes linearly separable.

The **kernel trick** allows us to compute dot products in that high-dimensional space *without* ever explicitly computing the mapping $\phi(x)$:

$$K(x, x') = \phi(x)^T \phi(x')$$

In [ ]:
# Generate non-linearly separable data (concentric circles)
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)

# Show that a linear SVM fails
fig, ax = plt.subplots(figsize=(8, 6))
svm_lin_fail = SVC(kernel='linear', C=1.0)
svm_lin_fail.fit(X_circles, y_circles)
plot_decision_boundary_2d(svm_lin_fail, X_circles, y_circles,
                          title='Linear SVM on Circles Data (fails!)', ax=ax)
plt.tight_layout()
plt.show()

print(f"Training accuracy: {svm_lin_fail.score(X_circles, y_circles):.2%}")

In [ ]:
# Lift the data to 3D: add x1^2 + x2^2 as a third feature
X_3d = np.column_stack([X_circles, X_circles[:, 0]**2 + X_circles[:, 1]**2])

fig = plt.figure(figsize=(14, 5))

# Left: original 2D data
ax1 = fig.add_subplot(121)
cmap_bold = ListedColormap(['#FF0000', '#0000FF'])
ax1.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap=cmap_bold,
            edgecolors='k', s=40, linewidths=0.5)
ax1.set_title('Original 2D space (not separable)', fontsize=13)
ax1.set_xlabel('$x_1$', fontsize=12)
ax1.set_ylabel('$x_2$', fontsize=12)
ax1.grid(True, alpha=0.3)

# Right: lifted 3D data
ax2 = fig.add_subplot(122, projection='3d')
colors_3d = np.array(['#FF0000', '#0000FF'])
for label in [0, 1]:
    mask = y_circles == label
    ax2.scatter(X_3d[mask, 0], X_3d[mask, 1], X_3d[mask, 2],
               c=colors_3d[label], edgecolors='k', s=40, linewidths=0.3,
               label=f'Class {label}', alpha=0.7)

# Draw a separating plane at z = threshold
threshold = 0.5 * (X_3d[y_circles == 0, 2].mean() + X_3d[y_circles == 1, 2].mean())
xx_plane, yy_plane = np.meshgrid(np.linspace(-1.5, 1.5, 10), np.linspace(-1.5, 1.5, 10))
zz_plane = np.full_like(xx_plane, threshold)
ax2.plot_surface(xx_plane, yy_plane, zz_plane, alpha=0.2, color='green')

ax2.set_title('Lifted 3D space (linearly separable!)', fontsize=13)
ax2.set_xlabel('$x_1$', fontsize=11)
ax2.set_ylabel('$x_2$', fontsize=11)
ax2.set_zlabel('$x_1^2 + x_2^2$', fontsize=11)
ax2.legend(fontsize=9)
ax2.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

print(f"Separating plane at z = {threshold:.3f}")

The mapping $\phi: (x_1, x_2) \mapsto (x_1, x_2, x_1^2 + x_2^2)$ lifts the circular data into 3D, where a simple plane can separate the classes.

The **kernel trick** lets us do this implicitly. Instead of computing $\phi(x)$ explicitly, we define a kernel function:

$$K(x, x') = \phi(x)^T \phi(x')$$

The SVM optimization only depends on dot products between data points, so we can replace every dot product $x^T x'$ with $K(x, x')$ and the algorithm works in the higher-dimensional space for free.

---
# Part 5: Common Kernels

The most commonly used kernel functions are:

| Kernel | Formula | Typical use |
|--------|---------|-------------|
| **Linear** | $K(x, x') = x^T x'$ | Linearly separable data, high-dimensional sparse data |
| **RBF (Gaussian)** | $K(x, x') = \exp(-\gamma \|x - x'\|^2)$ | General purpose, works well in most cases |
| **Polynomial** | $K(x, x') = (x^T x' + c)^d$ | Polynomial interaction features |

In [ ]:
# Generate make_moons data
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)

# Compare three kernels
kernels = ['linear', 'rbf', 'poly']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, kernel in zip(axes, kernels):
    svm_k = SVC(kernel=kernel, C=1.0, gamma='scale', degree=3)
    svm_k.fit(X_moons, y_moons)
    acc = svm_k.score(X_moons, y_moons)
    plot_decision_boundary_2d(svm_k, X_moons, y_moons,
                              title=f'Kernel: {kernel} (acc={acc:.0%})', ax=ax)

plt.suptitle('Decision Boundaries for Different Kernels (make_moons data)', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- **Linear**: cannot capture the curved boundary — underfits
- **RBF**: smoothly wraps around the two moons — excellent fit
- **Polynomial**: captures some curvature but may not match the shape perfectly

---
# Part 6: The Gamma Parameter (RBF Kernel)

For the RBF kernel $K(x, x') = \exp(-\gamma \|x - x'\|^2)$, the parameter $\gamma$ controls the **reach** of each support vector:

- **Small $\gamma$**: each support vector has wide influence $\rightarrow$ smooth, simple boundary (risk of underfitting)
- **Large $\gamma$**: each support vector has narrow influence $\rightarrow$ complex, wiggly boundary (risk of overfitting)

Intuitively, $\gamma = \frac{1}{2\sigma^2}$, where $\sigma$ is the radius of influence.

In [ ]:
# Effect of gamma on RBF SVM decision boundary
gamma_values = [0.1, 1, 10]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, gamma in zip(axes, gamma_values):
    svm_g = SVC(kernel='rbf', C=1.0, gamma=gamma)
    svm_g.fit(X_moons, y_moons)
    n_sv = svm_g.n_support_.sum()
    acc = svm_g.score(X_moons, y_moons)
    plot_decision_boundary_2d(svm_g, X_moons, y_moons,
                              title=f'gamma={gamma} ({n_sv} SVs, acc={acc:.0%})', ax=ax)

plt.suptitle('Effect of gamma on the RBF SVM Decision Boundary', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

**Key insight:**
- $\gamma = 0.1$: very smooth boundary, possibly underfitting (boundary is too simple)
- $\gamma = 1$: good balance between smoothness and complexity
- $\gamma = 10$: very complex boundary that tightly wraps around training points, likely overfitting

---
# Part 7: SVM for Regression (SVR)

Support Vector Machines can also be used for **regression** via **Support Vector Regression (SVR)**.

The idea: instead of maximizing a margin around a decision boundary, SVR fits a tube of width $\epsilon$ around the regression function. Points **inside the tube** have zero loss. Points **outside** contribute a loss proportional to their distance from the tube.

The optimization problem:

$$\min_{w, b} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^{n} (\xi_i + \xi_i^*)$$

subject to:
$$y_i - (w^T x_i + b) \leq \epsilon + \xi_i$$
$$(w^T x_i + b) - y_i \leq \epsilon + \xi_i^*$$
$$\xi_i, \xi_i^* \geq 0$$

In [ ]:
# Generate noisy sine data for regression
np.random.seed(42)
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + 0.3 * np.random.randn(80)

# Fit SVR with RBF kernel
epsilon = 0.3
svr = SVR(kernel='rbf', C=10, gamma='scale', epsilon=epsilon)
svr.fit(X_reg, y_reg)

# Predictions on a fine grid
X_plot = np.linspace(0, 5, 300).reshape(-1, 1)
y_pred = svr.predict(X_plot)

# Identify support vectors (points outside the epsilon-tube)
sv_indices = svr.support_

fig, ax = plt.subplots(figsize=(10, 6))

# Data points
ax.scatter(X_reg, y_reg, c='gray', edgecolors='k', s=40, linewidths=0.5,
           label='Training data', zorder=3)

# Highlight support vectors
ax.scatter(X_reg[sv_indices], y_reg[sv_indices], c='red', edgecolors='k',
           s=80, linewidths=1, label=f'Support vectors ({len(sv_indices)})', zorder=4)

# SVR prediction line
ax.plot(X_plot, y_pred, 'b-', linewidth=2.5, label='SVR prediction')

# Epsilon-tube
ax.fill_between(X_plot.ravel(), y_pred - epsilon, y_pred + epsilon,
                alpha=0.2, color='blue', label=f'$\\epsilon$-tube ($\\epsilon$={epsilon})')
ax.plot(X_plot, y_pred - epsilon, 'b--', linewidth=1, alpha=0.5)
ax.plot(X_plot, y_pred + epsilon, 'b--', linewidth=1, alpha=0.5)

ax.set_title('Support Vector Regression (SVR) with $\\epsilon$-insensitive tube', fontsize=14)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Number of support vectors: {len(sv_indices)} out of {len(X_reg)} data points")
print(f"Points inside the tube have ZERO loss.")

---
# Summary

## What We Learned

| Concept | Key Idea |
|---------|----------|
| **Maximum margin** | SVM picks the separating hyperplane that maximizes the distance to the nearest points |
| **Support vectors** | Only the points on the margin boundary influence the decision — the rest can be removed |
| **Soft margin ($C$)** | Allows misclassifications; large $C$ = tight fit, small $C$ = wide margin |
| **Kernel trick** | Maps data to higher dimensions implicitly via kernel functions |
| **RBF kernel ($\gamma$)** | Controls influence radius: small $\gamma$ = smooth, large $\gamma$ = complex |
| **SVR** | Regression with an $\epsilon$-insensitive tube around the prediction |

## Key Hyperparameters

| Parameter | Effect | Small value | Large value |
|-----------|--------|------------|-------------|
| $C$ | Regularization | More regularization, wider margin | Less regularization, tighter fit |
| $\gamma$ (RBF) | Influence radius | Smoother boundary | More complex boundary |
| $\epsilon$ (SVR) | Tube width | Narrow tube, more SVs | Wide tube, fewer SVs |
| `kernel` | Feature mapping | `linear` for simple data | `rbf` / `poly` for non-linear |

## When to Use SVMs

**Pros:**
- Effective in high-dimensional spaces
- Memory efficient (only stores support vectors)
- Versatile through different kernels
- Strong theoretical guarantees (margin maximization)

**Cons:**
- Does not scale well to very large datasets ($O(n^2)$ to $O(n^3)$)
- Sensitive to feature scaling (always standardize!)
- No direct probability estimates (use `probability=True` but it is slow)
- Choosing the right kernel and hyperparameters requires experimentation